# Re-register tables
This notebook reads the tables that are available in the `teehr` and `schema_evolution` directories ion S3 and reregisters the tables in Iceberg.  This should not be needed often.  Only if the catalog database is lost of we move to a new catalog.

In [1]:
import os
from pathlib import Path

import pandas as pd
import botocore.session
from botocore import UNSIGNED
from botocore.config import Config

from teehr.evaluation.spark_session_utils import create_spark_session

In [2]:
%%time
spark = create_spark_session()

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:✅ Spark local configuration successful!
INFO:teehr.evaluation.spark_session_utils:Setting Hadoop's default AWS credentials provider and AWS region
INFO:teehr.evaluation.spark_session_utils:🔑 Using AWS session token from boto3
INFO:teehr.evaluation.spark_session_utils:Configuring Iceberg catalogs...
INFO:teehr.evaluation.spark_session_utils:Discovered 8 Ivy package jars for spark.jars distribution
INFO:teehr.evaluation.spark_session_utils:⚙️ All settings applied. Creating Spark session...
INFO:teehr.evaluation.spark_session_utils:🎉 Spark session created successfully!


CPU times: user 111 ms, sys: 51.5 ms, total: 163 ms
Wall time: 28.4 s


In [5]:
spark.sql("USE iceberg").collect()

[]

In [3]:
bucket_name = "dev-teehr-iceberg-warehouse"

# Set up access for public S3 bucket
session = botocore.session.Session(profile="default")
creds = session.get_credentials()

s3 = session.create_client(
    's3',
    # config=Config(signature_version=UNSIGNED),
    region_name="us-east-2",
    # aws_access_key_id=creds.access_key,
    # aws_secret_access_key=creds.secret_key
)

INFO:botocore.credentials:Found credentials in shared credentials file: ~/.aws/credentials


Get a list of all table prefixes

In [4]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="teehr/",
    Delimiter='/',
    MaxKeys=100
)

table_prefixes = []
for prefix in response["CommonPrefixes"]:
    table_prefixes.append(prefix["Prefix"])

table_prefixes

['teehr/attributes/',
 'teehr/configuration_completeness/',
 'teehr/configurations/',
 'teehr/configurations_by_location/',
 'teehr/configurations_summary/',
 'teehr/fcst_joined_timeseries/',
 'teehr/fcst_metrics_by_lead_time_bins/',
 'teehr/fcst_metrics_by_location/',
 'teehr/forecast_metrics_by_location/',
 'teehr/grid_pixel_coverage_weights/',
 'teehr/grid_pixel_fractional_coverage/',
 'teehr/joined_forecast_timeseries/',
 'teehr/joined_simulation_timeseries/',
 'teehr/joined_timeseries/',
 'teehr/location_attributes/',
 'teehr/location_crosswalks/',
 'teehr/locations/',
 'teehr/nwmd_metrics_by_location/',
 'teehr/nwmd_metrics_by_location_test/',
 'teehr/primary_timeseries/',
 'teehr/secondary_timeseries/',
 'teehr/sim_joined_timeseries/',
 'teehr/sim_metrics_by_location/',
 'teehr/units/',
 'teehr/variables/',
 'teehr/workflow_state_joined_forecasts/']

For each table:
- Get a dataframe of metadata .json files and associated last modified times
- Get the path to the most recent metadata .json file
- Create the SQL to register that file
- Execute the SQL with the spark session

In [6]:
for table_prefix in table_prefixes:
    prefix = (f"{table_prefix}metadata/")
    response = s3.list_objects_v2(
        Bucket=bucket_name,
        Prefix=prefix,
        Delimiter='/',
        MaxKeys=100
    )
    meta_list = []
    for content in response["Contents"]:
        key = content["Key"]
        last_modified = content["LastModified"]
        if Path(key).suffix == ".json":
            meta_list.append(
                {"key": key, "last_modified": last_modified}
            )

    df = pd.DataFrame(meta_list)
    latest_indx = df.last_modified.idxmax()   ## --> NO!!
    latest_json = df.key[latest_indx]

    namespace = table_prefix.split("/")[0]
    table_name = table_prefix.split("/")[1]
    sql = (f"""
        CALL iceberg.system.register_table(
            table => '{namespace}.{table_name}',
            metadata_file => 's3://{bucket_name}/{latest_json}'
        )
    """)


    print()
    print(sql)

    spark.sql(sql)

    print(f"Registered table: {namespace}.{table_name}")

print("Registering complete!")



        CALL iceberg.system.register_table(
            table => 'teehr.attributes',
            metadata_file => 's3://dev-teehr-iceberg-warehouse/teehr/attributes/metadata/00007-db155b36-5ad4-42a8-abf2-bc419f20c50e.metadata.json'
        )
    
Registered table: teehr.attributes


        CALL iceberg.system.register_table(
            table => 'teehr.configuration_completeness',
            metadata_file => 's3://dev-teehr-iceberg-warehouse/teehr/configuration_completeness/metadata/00127-6fab919e-348d-45be-ad5e-b0b247e195f6.metadata.json'
        )
    
Registered table: teehr.configuration_completeness


        CALL iceberg.system.register_table(
            table => 'teehr.configurations',
            metadata_file => 's3://dev-teehr-iceberg-warehouse/teehr/configurations/metadata/00064-dce974b5-8edf-4740-b01c-94e5418097f5.metadata.json'
        )
    
Registered table: teehr.configurations


        CALL iceberg.system.register_table(
            table => 'teehr.configurations

In [7]:
response = s3.list_objects_v2(
    Bucket=bucket_name,
    Prefix="schema_evolution/",
    Delimiter='/',
    MaxKeys=100
)

table_prefixes = []
for prefix in response["CommonPrefixes"]:
    table_prefixes.append(prefix["Prefix"])

table_prefixes

['schema_evolution/schema_version_history/']

In [9]:
spark.sql("CREATE NAMESPACE schema_evolution")

DataFrame[]

In [10]:
for table_prefix in table_prefixes:
    prefix = (f"{table_prefix}metadata/")
    response = s3.list_objects_v2(
        Bucket=bucket_name,
        Prefix=prefix,
        Delimiter='/',
        MaxKeys=100
    )
    meta_list = []
    for content in response["Contents"]:
        key = content["Key"]
        last_modified = content["LastModified"]
        if Path(key).suffix == ".json":
            meta_list.append(
                {"key": key, "last_modified": last_modified}
            )

    df = pd.DataFrame(meta_list)
    latest_indx = df.last_modified.idxmax()   ## --> NO!!
    latest_json = df.key[latest_indx]

    namespace = table_prefix.split("/")[0]
    table_name = table_prefix.split("/")[1]
    sql = (f"""
        CALL iceberg.system.register_table(
            table => '{namespace}.{table_name}',
            metadata_file => 's3://{bucket_name}/{latest_json}'
        )
    """)


    print()
    print(sql)

    spark.sql(sql)

    print(f"Registered table: {namespace}.{table_name}")

print("Registering complete!")



        CALL iceberg.system.register_table(
            table => 'schema_evolution.schema_version_history',
            metadata_file => 's3://dev-teehr-iceberg-warehouse/schema_evolution/schema_version_history/metadata/00010-1f710c0d-18e9-470b-ab4f-2043dcd1d607.metadata.json'
        )
    
Registered table: schema_evolution.schema_version_history
Registering complete!


In [11]:
spark.stop()